In [6]:
import geopandas as gpd
from pyproj import Transformer
import matplotlib.pyplot as plt

gpd.options.io_engine = "pyogrio" #faster engine
markgeokemi_icpms = gpd.read_file(
    "C:/Projects/markgeokemi/raw_data/markgeokemi_regional.gpkg",
    layer = "moran_0063mm_hno3_icpms",
    use_arrow = True
)
#Drop rows with missing ID/coordinates, ensure SWEREF system 
markgeokemi_icpms_clean = markgeokemi_icpms.dropna(   
    subset=[
        "unikt_id", "ns", "ew", "geometry"
    ]).set_crs("EPSG:3006", allow_override=True) 
# save cleaned data to GeoPackage
markgeokemi_icpms_clean.to_file(
     "C:/Projects/markgeokemi/cleaned_data/markgeokemi_icpms_clean.gpkg",
    driver= "GPKG",
    use_arrow = True
)

#geographic bbox in WGS84; convert to SWEREF 99 TM
min_lat, max_lat = 59.20, 59.95
min_lon, max_lon = 17.30, 18.30
trans = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3006",
    always_xy = True
)
# convert bbox corners
min_ew, min_ns = trans.transform(min_lon, min_lat)  
max_ew, max_ns = trans.transform(max_lon, max_lat)

# keep valid (measured, >0) fe values within defined bbox
#0 → removes 0 (not analyzed
fe_stoch_upp = markgeokemi_icpms_clean[
    (markgeokemi_icpms_clean["fe_ppm"].notna()) & #removes missing values
    (markgeokemi_icpms_clean["fe_ppm"] > 0) & # > 0 removes not analysed & below detection
    (markgeokemi_icpms_clean["ew"].between(min_ew, max_ew)) &
    (markgeokemi_icpms_clean["ns"].between(min_ns, max_ns))
][["unikt_id", "ns", "ew", "fe_ppm", "geometry"]
].copy()

#summary stats for defined bbox
fe_stoch_upp_stat = fe_stoch_upp["fe_ppm"].describe()
median_fe_stoch_upp = fe_stoch_upp["fe_ppm"].median()
skew_fe_stoch_upp = fe_stoch_upp["fe_ppm"].skew()

print(fe_stoch_upp.shape)

C:\Users\Admin\anaconda3\Lib\site-packages\pyogrio\geopandas.py:346: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["geometry"] = shapely.from_wkb(wkb_values, on_invalid=on_invalid)


(535, 5)
